### The objective of this notebook is to filter the dataset for VEGRF-2 

The filteration will be based on orally bioavailable compounds

For a molecule to be a "lead" molecule it should follow Lipinski's rule of five (Ro5):
- Molecular weight (MWT) ≤ 500 Da. This is the size of the package. A very large molecule has trouble squeezing through the gut wall.
- LogP (octanol-water coefficient) ≤ 5. This measures how oily versus watery the molecule is. If it’s too oily, it won’t dissolve in the watery gut and blood and tends to get stuck in fat. If it’s too watery, it can’t pass through the oily cell membrane. You want a balance.
- Hydrogen bond donors (HBD) ≤ 5. These are the molecule’s “sticky hands” (mostly OH and NH groups) that cling to water.
- Hydrogen bond acceptors (HBA) ≤ 10. These are the sticky spots on the other side (mostly N and O atoms).

The last two matter because a molecule has to shed its water coat to slip through the cell membrane. The more sticky spots it has, the harder that is.

In [3]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, PandasTools


In [4]:
molecules = pd.read_csv("data/VEGFR-2.csv")
print(molecules.shape)
molecules.head(10)

(8954, 5)


,molecule_chembl_id,IC50,IC50_units,smiles,pIC50
0,CHEMBL5189340,0.023,nM,CN(C)c1ccc(/C=N/NC(=O)Cn2nc(Cc3ccc(Cl)cc3)c3cc...,10.638272
1,CHEMBL429743,0.030,nM,COc1cc2nccc(Oc3ccc4c(C(=O)Nc5ccc(Cl)cc5)cccc4c...,10.522879
2,CHEMBL5557999,0.100,nM,Cc1cc(C)nc(NS(=O)(=O)c2ccc(/N=N/c3ccc(C4=NCC(=...,10.000000
3,CHEMBL5411585,0.110,nM,COc1cc2c(Oc3ccc(NC(=O)Nc4ccc(C(F)(F)F)cc4)c(Cl...,9.958607
4,CHEMBL5186748,0.120,nM,CNC(=O)c1cc(Oc2ccc(NC(=O)c3nn(-c4ccc(Cl)cc4)cc...,9.920819
5,CHEMBL3586072,0.140,nM,CNC(=O)c1ccc(-c2ccc(NC(=O)Nc3cc(Br)cc(C(F)(F)F...,9.853872
6,CHEMBL3093581,0.140,nM,O=C(Nc1ccc(Oc2ccc3nc(NC(=O)C4CC4)cn3c2)c(F)c1)...,9.853872
7,CHEMBL3586071,0.150,nM,CNC(=O)c1cc(-c2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)...,9.823909
8,CHEMBL1289494,0.160,nM,COc1cc2nccc(Oc3ccc(NC(=O)Nc4cc(C)on4)c(Cl)c3)c...,9.795880
9,CHEMBL2348996,0.190,nM,Cc1cc(C(=O)Nc2cccc(Oc3ccc4nc(NC(=O)C5CC5)cn4n3...,9.721246


In [6]:
# writing the function that would filter the molecules based on the Lipinski's rule of five
def lipinski_filter(smiles):
    molecule = Chem.MolFromSmiles(smiles)
    
    molecular_weight = Descriptors.MolWt(molecule)
    logp = Descriptors.MolLogP(molecule)
    h_bond_donors = Descriptors.NumHDonors(molecule)
    h_bond_acceptors = Descriptors.NumHAcceptors(molecule)
    
    conditions = [
        molecular_weight <= 500,
        logp <= 5,
        h_bond_donors <= 5,
        h_bond_acceptors <= 10]
    
    ro5_fulfilled = sum(conditions) >= 3 # a molecule is considered to fulfill Lipinski's rule of five if it meets at least 3 out of the 4 conditions
    return pd.Series([molecular_weight, logp, h_bond_donors, h_bond_acceptors, ro5_fulfilled], 
                        index=["Molecular Weight", "LogP", "H-Bond Donors", "H-Bond Acceptors", "Ro5 Fulfilled"])

In [7]:
ro5_properties = molecules["smiles"].apply(lipinski_filter)
ro5_properties.head(10)

,Molecular Weight,LogP,H-Bond Donors,H-Bond Acceptors,Ro5 Fulfilled
0,473.964,3.85700,1,5,True
1,484.939,7.10320,1,5,True
2,574.066,5.77684,1,8,False
3,616.040,7.61160,2,7,False
4,489.919,3.99352,2,6,True
5,493.283,5.53350,3,3,True
6,523.524,5.01740,2,5,False
7,448.832,5.42440,3,3,True
8,454.870,5.63812,2,7,True
9,431.456,3.16432,2,6,True


In [8]:
molecule = pd.concat([molecules, ro5_properties], axis=1)
molecule.head(10)

,molecule_chembl_id,IC50,IC50_units,smiles,pIC50,Molecular Weight,LogP,H-Bond Donors,H-Bond Acceptors,Ro5 Fulfilled
0,CHEMBL5189340,0.023,nM,CN(C)c1ccc(/C=N/NC(=O)Cn2nc(Cc3ccc(Cl)cc3)c3cc...,10.638272,473.964,3.85700,1,5,True
1,CHEMBL429743,0.030,nM,COc1cc2nccc(Oc3ccc4c(C(=O)Nc5ccc(Cl)cc5)cccc4c...,10.522879,484.939,7.10320,1,5,True
2,CHEMBL5557999,0.100,nM,Cc1cc(C)nc(NS(=O)(=O)c2ccc(/N=N/c3ccc(C4=NCC(=...,10.000000,574.066,5.77684,1,8,False
3,CHEMBL5411585,0.110,nM,COc1cc2c(Oc3ccc(NC(=O)Nc4ccc(C(F)(F)F)cc4)c(Cl...,9.958607,616.040,7.61160,2,7,False
4,CHEMBL5186748,0.120,nM,CNC(=O)c1cc(Oc2ccc(NC(=O)c3nn(-c4ccc(Cl)cc4)cc...,9.920819,489.919,3.99352,2,6,True
5,CHEMBL3586072,0.140,nM,CNC(=O)c1ccc(-c2ccc(NC(=O)Nc3cc(Br)cc(C(F)(F)F...,9.853872,493.283,5.53350,3,3,True
6,CHEMBL3093581,0.140,nM,O=C(Nc1ccc(Oc2ccc3nc(NC(=O)C4CC4)cn3c2)c(F)c1)...,9.853872,523.524,5.01740,2,5,False
7,CHEMBL3586071,0.150,nM,CNC(=O)c1cc(-c2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)...,9.823909,448.832,5.42440,3,3,True
8,CHEMBL1289494,0.160,nM,COc1cc2nccc(Oc3ccc(NC(=O)Nc4cc(C)on4)c(Cl)c3)c...,9.795880,454.870,5.63812,2,7,True
9,CHEMBL2348996,0.190,nM,Cc1cc(C(=O)Nc2cccc(Oc3ccc4nc(NC(=O)C5CC5)cn4n3...,9.721246,431.456,3.16432,2,6,True


In [9]:
# to find the number of molecules that fulfill Lipinski's rule of five and those that do not compared to the total number of molecules in the dataset
total_molecules = molecule.shape[0]
ro5_fulfilled = molecule["Ro5 Fulfilled"].sum()
ro5_not_fulfilled = total_molecules - ro5_fulfilled

print(f"Total molecules: {total_molecules}")
print(f"Molecules that fulfill Ro5: {ro5_fulfilled}")
print(f"Molecules that do not fulfill Ro5: {ro5_not_fulfilled}")

Total molecules: 8954
Molecules that fulfill Ro5: 7261
Molecules that do not fulfill Ro5: 1693
